In [2]:
!pip install joblib
!pip install xgboost
!pip install catboost

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 281.2 kB/s eta 0:00:00m eta 0:00:010:00:10
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 313.7 kB/s eta 0:00:00m eta 0:00:010:00:01


In [ ]:


import numpy as np
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



dataset_files = {
    "welfake_unsup": "/content/drive/MyDrive/Mini/welfake_unsup_embeddings.npz",
    "welfake_sup": "/content/drive/MyDrive/Mini/welfake_sup_embeddings.npz",

    "fakenewsnet_unsup": "/content/drive/MyDrive/Mini/fakenewsnet_unsup_embeddings.npz",
    "fakenewsnet_sup": "/content/drive/MyDrive/Mini/fakenewsnet_sup_embeddings.npz",

    "fakepred_unsup": "/content/drive/MyDrive/Mini/fakepred_unsup_embeddings.npz",
    "fakepred_sup": "/content/drive/MyDrive/Mini/fakepred_sup_embeddings.npz",
}

# ============================================================

param_grid = {
    "DecisionTree": {
        "model": DecisionTreeClassifier(),
        "params": {"min_samples_split": [2, 5, 10]},
    },
    "RandomForest": {
        "model": RandomForestClassifier(),
        "params": {"n_estimators": [50, 100, 200]},
    },
    "SVM": {
        "model": SVC(kernel="linear"),
        "params": {"C": [0.1, 1, 10]},
    },
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000),
        "params": {"C": [1, 10, 100]},
    },
    "XGBoost": {
        "model": XGBClassifier(
            eval_metric="logloss", use_label_encoder=False,
            subsample=0.8, max_depth=3, learning_rate=0.1,
            n_estimators=100
        ),
        "params": {},
    },
    "CatBoost": {
        "model": CatBoostClassifier(
            depth=6, learning_rate=0.1, iterations=100,
            subsample=0.7, loss_function="Logloss", verbose=0
        ),
        "params": {},
    },
}


In [ ]:

# ============================================================
#  🚀 TRAINING LOOP FOR ALL DATASETS
# ============================================================

all_results = []

for dataset_name, file_path in dataset_files.items():
    print("\n====================================================")
    print(f"📌 TRAINING MODELS FOR DATASET: {dataset_name}")
    print("====================================================")

    # -----------------------------
    # Load embeddings
    # -----------------------------
    data = np.load(file_path)
    X = data["X"]
    y = data["y"]

    # -----------------------------
    # Train-test split
    # -----------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    # -----------------------------
    # Directory for models
    # -----------------------------
    save_dir = f"ML/trained_models/{dataset_name}"
    os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # Train each model
    # -----------------------------
    dataset_results = []

    for model_name, cfg in param_grid.items():
        print(f"\n🔹 Training {model_name} on {dataset_name}...")

        grid = GridSearchCV(cfg["model"], cfg["params"], cv=3,
                            scoring="accuracy", n_jobs=-1)
        grid.fit(X_train, y_train)
        y_pred = grid.predict(X_test)

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        # Save
        model_path = os.path.join(save_dir, f"{model_name}.pkl")
        joblib.dump(grid.best_estimator_, model_path)

        print(f"✅ Saved model to: {model_path}")
        print(f"📊 Accuracy={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f}")

        dataset_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Best Params": grid.best_params_,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1
        })

    # Save summary CSV for the dataset
    df = pd.DataFrame(dataset_results)
    df.to_csv(os.path.join(save_dir, "summary.csv"), index=False)

    all_results.extend(dataset_results)

# ============================================================
#  📌 SAVE GLOBAL SUMMARY
# ============================================================

final_df = pd.DataFrame(all_results)
final_df.to_csv("ML/all_dataset_model_summary.csv", index=False)

print("\n====================================================")
print("🎉 ALL MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("📂 Summary saved at: ML/all_dataset_model_summary.csv")
print("====================================================")

In [ ]:
# ================================================================
#   📊 GRAPH PLOTTING FOR ALL MODEL RESULTS
# ================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load the final summary
summary_path = "ML/all_dataset_model_summary.csv"
df = pd.read_csv(summary_path)

# Metrics to plot
metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]

# Create plots
for metric in metrics:
    plt.figure(figsize=(14, 6))
    sns.barplot(data=df, x="Model", y=metric, hue="Dataset")
    plt.title(f"{metric} Comparison Across Models and Datasets")
    plt.ylabel(metric)
    plt.xlabel("Model")
    plt.xticks(rotation=45)
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()


In [ ]:
# ================================================================
#    📌 CONFUSION MATRIX FOR EACH DATASET & MODEL
# ================================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import confusion_matrix

dataset_files = {
    "welfake_unsup": "/content/drive/MyDrive/Mini/welfake_unsup_embeddings.npz",
    "welfake_sup": "/content/drive/MyDrive/Mini/welfake_sup_embeddings.npz",
    "fakenewsnet_unsup": "/content/drive/MyDrive/Mini/fakenewsnet_unsup_embeddings.npz",
    "fakenewsnet_sup": "/content/drive/MyDrive/Mini/fakenewsnet_sup_embeddings.npz",
    "fakepred_unsup": "/content/drive/MyDrive/Mini/fakepred_unsup_embeddings.npz",
    "fakepred_sup": "/content/drive/MyDrive/Mini/fakepred_sup_embeddings.npz",
}


model_names = [
    "DecisionTree", "RandomForest", "SVM",
    "LogisticRegression", "XGBoost", "CatBoost"
]

for dataset_name, file_path in dataset_files.items():

    print("\n====================================================")
    print(f"📌 CONFUSION MATRICES FOR DATASET: {dataset_name}")
    print("====================================================")

    # Load dataset
    data = np.load(file_path)
    X = data["X"]
    y = data["y"]

    # Train-test split (same as before so predictions match)
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model_dir = f"ML/trained_models/{dataset_name}"

    for model_name in model_names:
        model_path = os.path.join(model_dir, f"{model_name}.pkl")

        if not os.path.exists(model_path):
            print(f"⚠ Model not found: {model_path}")
            continue

        # Load model
        model = joblib.load(model_path)

        # Predict
        y_pred = model.predict(X_test)

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)

        # Plot
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
        plt.title(f"Confusion Matrix - {dataset_name} - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.show()
